<pre>
- Ozônio
</pre>

Unidade nativa: kg/kg-¹ (razão de mistura em massa / mass mixing ratio)
O que significa: Quilogramas do gás poluidor por quilograma de ar.


In [ ]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [ ]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("Ozonio")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
def get_EAC4_(year, client):
    dataset = "cams-global-reanalysis-eac4"
    request = {
        "variable": [
            "ozone"
        ],
        "pressure_level": ["1000"],
        "date": [f"{year}-12-01/{year}-12-31"],
        "time": ["06:00"],
        "data_format": "netcdf",
        "area": [6, -74, -35, -34]
    }
    ret_download = client.retrieve(dataset, request).download()

    return ret_download

def convert_nc_to_spark_dataframe(path, file_name):
    with xr.open_dataset(f"{path}\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_ozonio = spark.createDataFrame(df_dask_c)

    return df_ozonio

def convert_unit(df_ozonio):
    # Converte o valor do monoxido de carbodo de kg/kg-¹ para ppb (partes por bilhão)

    M_AR = 28.9644 # g/mol
    M_O3 = 47.9982 # g/mol
    FATOR_CONVERSAO = (M_AR / M_O3) * 1e9  # ~ 1.03407e9

    drop_cols = ["valid_time", "pressure_level", "go3"]

    df_ozonio_ppb = \
        (df_ozonio
            .withColumns({"data_medicao": F.col("valid_time").cast("date")
                        ,"indicador": F.lit("Poluição do ar - O₃ (ppb)") 
                        ,"valor": (F.col("go3") * F.lit(FATOR_CONVERSAO)).cast("double")
                        ,"unidade_medida": F.lit("ppb")})
            .drop(*drop_cols)

        )

    return df_ozonio_ppb

def write_data_csv(df_ozonio_ppb, write_path, file_name):
    
    df_ozonio_ppb.toPandas().to_csv(f"{write_path}\{file_name}")

In [ ]:
years_process = [2005, 2006, 2007, 2008, 2009
                ,2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019
                ,2020, 2021, 2022, 2023, 2024, 2025]

# client = cdsapi.Client(
#     url = "https://ads.atmosphere.copernicus.eu/api",
#     key = "34161618-bf6b-41ca-9272-50b917f789b9"
# )

for year in years_process:

    # ret_download = get_EAC4_(year, client)
    print(f"{year} - Convert unit mean and save NC files to CSV", end = "")

    df_ozonio = \
        convert_nc_to_spark_dataframe(f"{DATA_PATH_ROOT}\EAC4-poluicao", f"EAC4_go3_{year}.nc")

    df_ozonio_ppb = convert_unit(df_ozonio)

    csv_path      = r"{DATA_PATH_ROOT}\EAC4-poluicao\arquivos_csv\go3".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"EAC4_co_{year}.csv"

    write_data_csv(df_ozonio_ppb, csv_path, csv_file_name)

    # os.rename(ret_download, f"{DATA_PATH_ROOT}\EAC4-poluicao\EAC4_go3_{year}.nc")


    print(f" - completed","\n")